# Day 7: Pandas Data Manipulation & Aggregation 
This notebook is divided into two parts:
1. **Core Practice:** Experimenting with filtering, sorting, grouping, and creating dataframes from scratch.
2. **Final Practise Project:** A structured workflow to generate a polished region-wise sales report from raw CSV data.

---
# Part 1: Core Pandas Practice
Practicing foundational data wrangling techniques using both external CSV files and manually created dictionaries.

In [14]:
import pandas as pd

# 1. Securely load the dataset with error handling
try:
    df_practice = pd.read_csv("sales_data.csv")
    print("CSV loaded successfully")
except FileNotFoundError:
    print("File not found")

CSV loaded successfully


In [15]:
# 2. Clean the 'Sales' column: Remove '₹' and ',' then convert to integer
df_practice['Sales'] = df_practice['Sales'].str.replace('₹', '').str.replace(',', '').astype(int)

print("\n--- Cleaned Data Preview ---")
display(df_practice.head())


--- Cleaned Data Preview ---


,OrderID,Region,Product,Sales,Quantity,Month
0,1001,North,Laptop,45000,2,Jan
1,1002,South,Phone,18000,5,Jan
2,1003,East,Tablet,22000,3,Feb
3,1004,North,Laptop,45000,1,Feb
4,1005,West,Phone,18000,4,Mar


In [16]:
# --- FILTERING ---
print("--- Sales greater than 30000 ---")
display(df_practice[df_practice["Sales"] > 30000])

print("\n--- Jan month Sales data ---")
display(df_practice[df_practice["Month"] == "Jan"])

--- Sales greater than 30000 ---


,OrderID,Region,Product,Sales,Quantity,Month
0,1001,North,Laptop,45000,2,Jan
3,1004,North,Laptop,45000,1,Feb
6,1007,East,Laptop,45000,3,Mar



--- Jan month Sales data ---


,OrderID,Region,Product,Sales,Quantity,Month
0,1001,North,Laptop,45000,2,Jan
1,1002,South,Phone,18000,5,Jan


In [17]:
# --- GROUP BY & AGGREGATION ---
print("\n--- Sales from Jan to Mar (Sum by Region) ---")
display(df_practice.groupby("Region")["Sales"].sum().sort_values(ascending=False))

print("\n--- Highest sales grouped by month (Descending) ---")
display(df_practice.groupby("Month")["Sales"].sum().sort_values(ascending=False))


--- Sales from Jan to Mar (Sum by Region) ---


Region
North    90000
East     67000
South    40000
West     18000
Name: Sales, dtype: int64


--- Highest sales grouped by month (Descending) ---


Month
Mar    85000
Feb    67000
Jan    63000
Name: Sales, dtype: int64

In [18]:
# --- CREATING DATAFRAMES FROM DICTIONARIES ---
sales_data_dict = {
    "Region": ["North", "South", "East", "West", "North", "South", "East", "West", "North", "South"],
    "SalesPerson": ["Rahul", "Priya", "Arun", "Meena", "Karthik", "Divya", "Vijay", "Anu", "Suresh", "Kavya"],
    "Product": ["Laptop", "Mobile", "Laptop", "Tablet", "Mobile", "Laptop", "Tablet", "Laptop", "Mobile", "Tablet"],
    "Sales": [45000, 60000, 35000, 40000, 55000, 70000, 30000, 65000, 50000, 45000]
}

df_manual = pd.DataFrame(sales_data_dict)

# --- ADVANCED SORTING ---
# Filter active sales and sort by Region (Ascending) and Sales (Descending)
df_manual_active = df_manual[df_manual["Sales"] > 0]
df_manual_sorted = df_manual_active.sort_values(['Region', 'Sales'], ascending=[True, False])

print("--- Multi-level Sorted Manual DataFrame ---")
display(df_manual_sorted)

--- Multi-level Sorted Manual DataFrame ---


,Region,SalesPerson,Product,Sales
2,East,Arun,Laptop,35000
6,East,Vijay,Tablet,30000
4,North,Karthik,Mobile,55000
8,North,Suresh,Mobile,50000
0,North,Rahul,Laptop,45000
5,South,Divya,Laptop,70000
1,South,Priya,Mobile,60000
9,South,Kavya,Tablet,45000
7,West,Anu,Laptop,65000
3,West,Meena,Tablet,40000


---
# Part 2: Practice Project - Region-wise Sales Report
**Goal:** Create a complete sales report showing total sales, average sales, order count, and the top-performing product for each region using the original CSV dataset.

In [19]:
# Step 1: Load & Filter Active Orders
# (Reloading fresh to ensure a clean state for the project)
df_project = pd.read_csv('sales_data.csv')
df_project['Sales'] = df_project['Sales'].str.replace('₹', '').str.replace(',', '').astype(int)
df_active = df_project[df_project['Sales'] > 0]
df_active

,OrderID,Region,Product,Sales,Quantity,Month
0,1001,North,Laptop,45000,2,Jan
1,1002,South,Phone,18000,5,Jan
2,1003,East,Tablet,22000,3,Feb
3,1004,North,Laptop,45000,1,Feb
4,1005,West,Phone,18000,4,Mar
5,1006,South,Tablet,22000,2,Mar
6,1007,East,Laptop,45000,3,Mar


In [20]:
# Step 2: Sort by Region then Sales
df_sorted = df_active.sort_values(['Region', 'Sales'], ascending=[True, False])
df_sorted

,OrderID,Region,Product,Sales,Quantity,Month
6,1007,East,Laptop,45000,3,Mar
2,1003,East,Tablet,22000,3,Feb
0,1001,North,Laptop,45000,2,Jan
3,1004,North,Laptop,45000,1,Feb
5,1006,South,Tablet,22000,2,Mar
1,1002,South,Phone,18000,5,Jan
4,1005,West,Phone,18000,4,Mar


In [21]:
# Step 3: Group by Region & Aggregate
report = df_sorted.groupby('Region')['Sales'].agg(['sum', 'mean', 'count'])
report

,sum,mean,count
Region,,,
East,67000,33500.0,2
North,90000,45000.0,2
South,40000,20000.0,2
West,18000,18000.0,1


In [22]:
# Step 4: Rename Columns
report.columns = ['Total Sales', 'Avg Sale', 'Order Count']
report['Avg Sale'] = report['Avg Sale'].round(2) # Clean up the decimal places
report

,Total Sales,Avg Sale,Order Count
Region,,,
East,67000,33500.0,2
North,90000,45000.0,2
South,40000,20000.0,2
West,18000,18000.0,1


In [23]:
# Find the Top Product per Region
top_products = df_active.groupby('Region')['Product'].agg(lambda x: x.mode()[0]).reset_index()
top_products.columns = ['Region', 'Top Product']
top_products

,Region,Top Product
0,East,Laptop
1,North,Laptop
2,South,Phone
3,West,Phone


In [24]:
# Merge the top products into our main report
final_report = pd.merge(report, top_products, on='Region')
final_report.set_index('Region', inplace=True)
final_report

,Total Sales,Avg Sale,Order Count,Top Product
Region,,,,
East,67000,33500.0,2,Laptop
North,90000,45000.0,2,Laptop
South,40000,20000.0,2,Phone
West,18000,18000.0,1,Phone


In [25]:
print("--- Final Region-wise Sales Report ---")
display(final_report)

--- Final Region-wise Sales Report ---


,Total Sales,Avg Sale,Order Count,Top Product
Region,,,,
East,67000,33500.0,2,Laptop
North,90000,45000.0,2,Laptop
South,40000,20000.0,2,Phone
West,18000,18000.0,1,Phone
